# WOS Meeting Fine-Tuning on Colab

This notebook adapts the original public-dataset QLoRA SFT recipe to Google Colab for the **meeting summarization / action-item / decision extraction** model. It keeps the same overall method because it will work, but it now targets a **time-constrained, sub-32B model set** and performs explicit Hugging Face access checks before any training begins.

## Candidate bases in this notebook

All three candidates are from **different model families** and all are **below 32B**. Auto-selection exists only for one-off compatibility checks. It is **not** the final assignment workflow.

1. `google/gemma-2-27b-it` - strongest meeting candidate here while staying below 32B.
2. `mistralai/Mistral-Small-3.1-24B-Instruct-2503` - high-quality fallback from a different family.
3. `meta-llama/Llama-3.1-8B-Instruct` - smallest gated fallback so the notebook still runs on smaller GPU tiers.

To satisfy the assignment, you must complete **all three meeting runs**: `gemma_2_27b`, `mistral_small_24b`, and `llama_3_1_8b`. Together with the coding notebook, that means **six total fine-tuning runs**.

## Before you run

1. Accept the Meta license for Llama at `https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct`
2. Accept the Google license for Gemma at `https://huggingface.co/google/gemma-2-27b-it`
3. Create a Colab secret named `HF_TOKEN` with a Hugging Face token if you want the gated Llama or Gemma variants

## Audit of the original training method

- **Yes, it will work** for a meeting model: chat-format SFT on a causal LM is a valid fine-tuning method for summarization and extraction tasks.
- The weakest parts of the original script for meeting data are **`max_seq_length = 1024`**, **`packing = True`**, and the lack of any leakage guard around synthetic action-item variants.
- This notebook keeps the same core SFT method, but defaults to a longer context window and turns packing off for safer first runs.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import random
import re
import subprocess

try:
    drive = importlib.import_module("google.colab.drive")
except ModuleNotFoundError as exc:
    raise SystemExit("This notebook must run inside Google Colab.") from exc

drive.mount("/content/drive", force_remount=False)

WOS_DATA_ROOT = Path("/content/drive/MyDrive/wos_data/meeting")
DATA_DIR = WOS_DATA_ROOT / "datasets"
RUNS_DIR = WOS_DATA_ROOT / "runs"
ARTIFACTS_DIR = WOS_DATA_ROOT / "artifacts"

for path in (DATA_DIR, RUNS_DIR, ARTIFACTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "wos_data_root": str(WOS_DATA_ROOT),
    "data_dir": str(DATA_DIR),
    "runs_dir": str(RUNS_DIR),
    "artifacts_dir": str(ARTIFACTS_DIR),
}, indent=2))

In [ ]:
MEETING_MODEL_OPTIONS = {
    "gemma_2_27b": {
        "model_name": "google/gemma-2-27b-it",
        "family": "Gemma",
        "gated": True,
        "min_vram_gib": 28.0,
        "max_seq_length": 3072,
        "learning_rate": 1e-4,
        "gradient_accumulation_steps": 16,
        "access_url": "https://huggingface.co/google/gemma-2-27b-it",
        "note": "Strongest meeting candidate here while staying below 32B.",
    },
    "mistral_small_24b": {
        "model_name": "mistralai/Mistral-Small-3.1-24B-Instruct-2503",
        "family": "Mistral",
        "gated": False,
        "min_vram_gib": 28.0,
        "max_seq_length": 3072,
        "learning_rate": 1e-4,
        "gradient_accumulation_steps": 16,
        "access_url": "https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503",
        "note": "High-quality fallback from a different family.",
    },
    "llama_3_1_8b": {
        "model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "family": "Llama",
        "gated": True,
        "min_vram_gib": 14.0,
        "max_seq_length": 3072,
        "learning_rate": 1.5e-4,
        "gradient_accumulation_steps": 16,
        "access_url": "https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct",
        "note": "Smallest gated fallback so the notebook still runs on smaller GPU tiers.",
    },
}

REQUIRED_MODEL_KEYS = ["gemma_2_27b", "mistral_small_24b", "llama_3_1_8b"]
BASE_MODEL_KEY = REQUIRED_MODEL_KEYS[0]  # Change this for each of the 3 required meeting runs. Use "auto" only for quick compatibility checks.

def detect_gpu():
    try:
        raw = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            text=True,
        ).strip()
    except Exception as exc:
        raise SystemExit("A GPU Colab runtime is required before training.") from exc

    rows = []
    for line in raw.splitlines():
        if not line.strip():
            continue
        name, memory = [part.strip() for part in line.split(",", 1)]
        mib = float(re.sub(r"[^0-9.]", "", memory))
        rows.append({
            "name": name,
            "memory_gib": round(mib / 1024.0, 2),
        })
    if not rows:
        raise SystemExit("No GPU was reported by nvidia-smi.")
    return rows[0]

def choose_model_key(requested_key: str, gpu_info: dict) -> str:
    if requested_key != "auto":
        if requested_key not in MEETING_MODEL_OPTIONS:
            raise KeyError(f"Unknown BASE_MODEL_KEY: {requested_key}")
        return requested_key

    for key, option in MEETING_MODEL_OPTIONS.items():
        if gpu_info["memory_gib"] >= option["min_vram_gib"]:
            return key
    raise SystemExit(
        f"No configured meeting model fits this runtime. GPU memory: {gpu_info['memory_gib']:.1f} GiB."
    )

gpu = detect_gpu()
SELECTED_MODEL_KEY = choose_model_key(BASE_MODEL_KEY, gpu)
profile = MEETING_MODEL_OPTIONS[SELECTED_MODEL_KEY]

print(json.dumps({
    "gpu": gpu,
    "requested_model_key": BASE_MODEL_KEY,
    "selected_model_key": SELECTED_MODEL_KEY,
    "required_model_keys": REQUIRED_MODEL_KEYS,
    "assignment_total_model_runs": 6,
    "available_models": MEETING_MODEL_OPTIONS,
    "profile": profile,
}, indent=2))

In [ ]:
get_ipython().run_line_magic("pip", "install --upgrade pip")
get_ipython().run_line_magic("pip", "install \"accelerate>=1.4.0,<2.0.0\" \"bitsandbytes==0.49.2\" \"datasets>=4.8.0,<5.0.0\" \"huggingface-hub>=0.36.0,<1.0.0\" \"peft>=0.17.0,<1.0.0\" \"rouge-score>=0.1.2\" \"safetensors>=0.7.0\" \"sentencepiece>=0.2.0\" \"transformers>=4.57.0,<4.59.0\" \"trl==0.24.0\"")

import importlib.metadata as md
for package in ["transformers", "trl", "peft", "datasets", "accelerate", "bitsandbytes"]:
    print(f"{package}: {md.version(package)}")

In [ ]:
from huggingface_hub import HfApi
from jinja2.exceptions import TemplateError
from transformers import AutoTokenizer

HF_TOKEN = None
try:
    userdata = importlib.import_module("google.colab.userdata")
except ModuleNotFoundError:
    userdata = None

if userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if profile.get("gated") and not HF_TOKEN:
    raise SystemExit(
        f"Set a Colab secret named HF_TOKEN before continuing with {profile['model_name']}. This model is gated."
    )
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

api = HfApi()
model_info_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
try:
    api.model_info(profile["model_name"], **model_info_kwargs)
except Exception as exc:
    if profile.get("gated"):
        raise RuntimeError(
            f"No access to {profile['model_name']}. Accept the license at {profile['access_url']} and rerun this cell. Original error: {exc}"
        ) from exc
    raise RuntimeError(
        f"Unable to reach {profile['model_name']} at {profile['access_url']}. Original error: {exc}"
    ) from exc

tokenizer = AutoTokenizer.from_pretrained(
    profile["model_name"],
    token=HF_TOKEN or None,
    trust_remote_code=True,
)

def tokenizer_has_chat_template(active_tokenizer) -> bool:
    template = getattr(active_tokenizer, "chat_template", None) or ""
    return bool(template.strip())

def uses_mistral_v7_fallback(active_tokenizer) -> bool:
    if tokenizer_has_chat_template(active_tokenizer):
        return False
    family = str(profile.get("family", "")).strip().lower()
    model_name = str(profile.get("model_name", "")).strip().lower()
    tokenizer_name = str(getattr(active_tokenizer, "name_or_path", "")).strip().lower()
    return family == "mistral" or "mistral" in model_name or "mistral" in tokenizer_name

def prepare_tokenizer_for_chat(active_tokenizer):
    if active_tokenizer.pad_token is None:
        active_tokenizer.pad_token = active_tokenizer.eos_token
    active_tokenizer.padding_side = "right"
    if active_tokenizer.eos_token_id is None:
        raise RuntimeError(f"{profile['model_name']} does not expose an eos_token_id.")
    if tokenizer_has_chat_template(active_tokenizer) or uses_mistral_v7_fallback(active_tokenizer):
        return active_tokenizer
    raise RuntimeError(
        f"{profile['model_name']} does not expose a usable chat template, and this notebook only provides a fallback for Mistral instruct models."
    )

tokenizer = prepare_tokenizer_for_chat(tokenizer)

def normalize_messages_for_template(messages: list[dict]) -> list[dict]:
    normalized = []
    for message in messages:
        role = str(message.get("role", "")).strip()
        content = message.get("content", "")
        if not isinstance(content, str):
            raise TypeError(
                f"Expected string chat content for role {role!r}, got {type(content).__name__}."
            )
        normalized.append({"role": role, "content": content})
    return normalized

def fold_system_message_into_user(messages: list[dict]) -> list[dict]:
    normalized = normalize_messages_for_template(messages)
    system_chunks = [
        message["content"].strip()
        for message in normalized
        if message["role"] == "system" and message["content"].strip()
    ]
    remaining = [dict(message) for message in normalized if message["role"] != "system"]
    if not system_chunks:
        return [dict(message) for message in normalized]
    merged_system = "\n\n".join(system_chunks).strip()
    if not remaining:
        return [{"role": "user", "content": merged_system}]
    first_message = dict(remaining[0])
    if first_message["role"] == "user":
        user_content = first_message.get("content", "")
        first_message["content"] = f"{merged_system}\n\n{user_content}".strip()
        remaining[0] = first_message
    else:
        remaining.insert(0, {"role": "user", "content": merged_system})
    return remaining

def render_mistral_v7_prompt(messages: list[dict], add_generation_prompt: bool = False) -> str:
    del add_generation_prompt  # The V7 prompt already ends in the correct generation position after a user turn.
    normalized = normalize_messages_for_template(messages)
    system_chunks = [
        message["content"].strip()
        for message in normalized
        if message["role"] == "system" and message["content"].strip()
    ]
    dialogue = [message for message in normalized if message["role"] != "system"]
    if not dialogue:
        return ""

    parts = []
    pending_system = "\n\n".join(system_chunks).strip()
    for message in dialogue:
        role = message["role"]
        content = message["content"]
        if role == "user":
            turn = "<s>"
            if pending_system:
                turn += f"[SYSTEM_PROMPT]{pending_system}[/SYSTEM_PROMPT]"
                pending_system = ""
            turn += f"[INST]{content}[/INST]"
            parts.append(turn)
        elif role == "assistant":
            parts.append(f"{content}</s>")
        else:
            raise ValueError(f"Unsupported role {role!r} for the Mistral fallback template.")
    return "".join(parts)

def apply_chat_template_fallback(active_tokenizer, messages, tokenize=False, add_generation_prompt=False, **kwargs):
    rendered = render_mistral_v7_prompt(messages, add_generation_prompt=add_generation_prompt)
    if not tokenize:
        return rendered
    encoded = active_tokenizer(rendered, add_special_tokens=False, **kwargs)
    return encoded["input_ids"]

def apply_chat_template_safe(active_tokenizer, messages, **kwargs):
    prepared_messages = normalize_messages_for_template(messages)
    if uses_mistral_v7_fallback(active_tokenizer):
        return apply_chat_template_fallback(active_tokenizer, prepared_messages, **kwargs)
    try:
        return active_tokenizer.apply_chat_template(prepared_messages, **kwargs)
    except TemplateError as exc:
        if "System role not supported" not in str(exc):
            raise
        repaired_messages = fold_system_message_into_user(prepared_messages)
        return active_tokenizer.apply_chat_template(
            repaired_messages,
            **kwargs,
        )

sample_messages = [
    {"role": "system", "content": "You are a careful meeting assistant."},
    {"role": "user", "content": "Summarize this meeting and list action items."},
    {"role": "assistant", "content": "Summary: ...\nAction items: ..."},
]
rendered = apply_chat_template_safe(tokenizer, sample_messages, tokenize=False, add_generation_prompt=False)
generation_prompt = apply_chat_template_safe(tokenizer, sample_messages[:2], tokenize=False, add_generation_prompt=True)
if not rendered.strip() or not generation_prompt.strip():
    raise RuntimeError(f"Chat template rendering failed for {profile['model_name']}.")
if sample_messages[1]["content"] not in rendered:
    raise RuntimeError(f"Rendered prompt for {profile['model_name']} did not preserve the user turn.")

chat_template_source = "native" if tokenizer_has_chat_template(tokenizer) else "mistral_v7_fallback"
print(json.dumps({
    "model_name": profile["model_name"],
    "family": profile["family"],
    "selected_model_key": SELECTED_MODEL_KEY,
    "gated": profile.get("gated", False),
    "eos_token": tokenizer.eos_token,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token": tokenizer.pad_token,
    "pad_token_id": tokenizer.pad_token_id,
    "chat_template_present": True,
    "chat_template_source": chat_template_source,
    "access_url": profile["access_url"],
    "rendered_preview": rendered[:700],
    "generation_preview": generation_prompt[:700],
}, indent=2))

In [ ]:
import hashlib
from collections import Counter

from datasets import load_dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)

SYSTEM_PROMPT = (
    "You are WOS Meeting, an expert meeting intelligence assistant. "
    "You excel at summarizing meeting transcripts, extracting action items, "
    "identifying key decisions, and answering questions about meeting content."
)

TRAIN_JSONL = DATA_DIR / "train.jsonl"
TRAIN_SPLIT_JSONL = DATA_DIR / "train_split.jsonl"
EVAL_SPLIT_JSONL = DATA_DIR / "eval_split.jsonl"
DATASET_MANIFEST = DATA_DIR / "manifest.json"

def to_record(prompt: str, response: str, source: str) -> dict:
    return {
        "source": source,
        "conversations": [
            {"from": "system", "value": SYSTEM_PROMPT},
            {"from": "human", "value": prompt},
            {"from": "gpt", "value": response},
        ],
    }

def download_dialogsum() -> list[dict]:
    print("Downloading DialogSum...")
    ds = load_dataset("knkarthick/dialogsum", split="train")
    samples = []
    for row in tqdm(ds, desc="DialogSum"):
        dialogue = row.get("dialogue", "")
        summary = row.get("summary", "")
        if not dialogue or not summary:
            continue
        prompt = (
            "Please summarize the following conversation and extract any action items:\n\n"
            f"{dialogue}"
        )
        samples.append(to_record(prompt, summary, "dialogsum"))
    return samples

def download_meetingbank() -> list[dict]:
    print("Downloading MeetingBank...")
    try:
        ds = load_dataset("huuuyeah/meetingbank", split="train")
    except Exception as exc:
        print(f"MeetingBank load failed: {exc} -- skipping")
        return []
    samples = []
    for row in tqdm(ds, desc="MeetingBank"):
        transcript = row.get("transcript", "") or row.get("meeting_transcripts", "")
        summary = row.get("summary", "")
        if not transcript or not summary:
            continue
        if len(transcript) > 8000:
            transcript = transcript[:8000] + "\n[transcript truncated]"
        prompt = (
            "Below is a meeting transcript. Please provide:\n"
            "1. A concise summary\n"
            "2. Key decisions made\n"
            "3. Action items with owners (if mentioned)\n\n"
            f"TRANSCRIPT:\n{transcript}"
        )
        samples.append(to_record(prompt, summary, "meetingbank"))
    return samples

def download_qmsum() -> list[dict]:
    print("Downloading QMSum...")
    try:
        ds = load_dataset("yale-nlp/QMSum", split="train")
    except Exception:
        try:
            ds = load_dataset("pszemraj/qmsum-cleaned", split="train")
        except Exception as exc:
            print(f"QMSum load failed: {exc} -- skipping")
            return []
    samples = []
    for row in tqdm(ds, desc="QMSum"):
        meeting = row.get("meeting", row.get("transcript", ""))
        query = row.get("query", row.get("question", ""))
        answer = row.get("answer", row.get("summary", ""))
        if not meeting or not answer:
            continue
        if len(meeting) > 6000:
            meeting = meeting[:6000] + "\n[transcript truncated]"
        prompt = (
            f"MEETING TRANSCRIPT:\n{meeting}\n\nQUESTION: {query}"
            if query
            else f"MEETING TRANSCRIPT:\n{meeting}\n\nSummarize this meeting."
        )
        samples.append(to_record(prompt, answer, "qmsum"))
    return samples

def add_action_item_samples(base_samples: list[dict], n: int = 2000) -> list[dict]:
    extras = []
    pool = random.sample(base_samples, min(n, len(base_samples)))
    for item in pool:
        original_human = item["conversations"][1]["value"]
        original_gpt = item["conversations"][2]["value"]
        prompt = original_human.replace(
            "Please summarize",
            "Extract all action items from",
        ).replace(
            "Below is a meeting transcript. Please provide:",
            "List only the action items from this meeting:",
        )
        if prompt == original_human:
            continue
        extras.append(to_record(prompt, f"Action items extracted:\n{original_gpt}", "synthetic_action_items"))
    return extras

dialogsum = download_dialogsum()
meetingbank = download_meetingbank()
qmsum = download_qmsum()
all_samples = dialogsum + meetingbank + qmsum
all_samples += add_action_item_samples(all_samples, n=2000)
random.shuffle(all_samples)

split_index = int(len(all_samples) * 0.95)
train_split = all_samples[:split_index]
eval_split = all_samples[split_index:]

for target_path, records in [
    (TRAIN_JSONL, all_samples),
    (TRAIN_SPLIT_JSONL, train_split),
    (EVAL_SPLIT_JSONL, eval_split),
]:
    with target_path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

manifest = {
    "seed": SEED,
    "total_samples": len(all_samples),
    "train_samples": len(train_split),
    "eval_samples": len(eval_split),
    "source_counts": dict(Counter(record["source"] for record in all_samples)),
    "train_jsonl": str(TRAIN_JSONL),
    "train_split_jsonl": str(TRAIN_SPLIT_JSONL),
    "eval_split_jsonl": str(EVAL_SPLIT_JSONL),
}
DATASET_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def row_hash(row: dict) -> str:
    payload = json.dumps(row["conversations"], sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

train_rows = read_jsonl(TRAIN_SPLIT_JSONL)
eval_rows = read_jsonl(EVAL_SPLIT_JSONL)
train_hashes = {row_hash(row) for row in train_rows}
eval_hashes = {row_hash(row) for row in eval_rows}

def prompt_length(row: dict) -> int:
    return len(row["conversations"][1]["value"])

def response_length(row: dict) -> int:
    return len(row["conversations"][2]["value"])

summary = {
    "exact_train_eval_overlap": len(train_hashes & eval_hashes),
    "train_source_counts": dict(Counter(row["source"] for row in train_rows)),
    "eval_source_counts": dict(Counter(row["source"] for row in eval_rows)),
    "max_train_prompt_chars": max(prompt_length(row) for row in train_rows),
    "max_eval_prompt_chars": max(prompt_length(row) for row in eval_rows),
    "max_train_response_chars": max(response_length(row) for row in train_rows),
    "max_eval_response_chars": max(response_length(row) for row in eval_rows),
    "sample_prompt_preview": train_rows[0]["conversations"][1]["value"][:700],
    "sample_response_preview": train_rows[0]["conversations"][2]["value"][:500],
}
print(json.dumps(summary, indent=2))
if summary["exact_train_eval_overlap"]:
    raise RuntimeError("Exact duplicate leakage detected between train and eval splits.")

In [ ]:
START_TRAINING_NOW = False
MERGE_AFTER_TRAIN = False
PUSH_ADAPTER_TO_HUB = False
HF_MODEL_REPO = None  # example: your-handle/wos-meeting-llama-3-3-70b-lora

MAX_TRAIN_SAMPLES = 6000
NUM_TRAIN_EPOCHS = 1.0
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
PACKING = False
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2
REPORT_TO = "none"

OUTPUT_DIR = RUNS_DIR / f"wos-meeting-{SELECTED_MODEL_KEY}-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
from datasets import Dataset
from huggingface_hub import HfApi
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def load_jsonl_dataset(path: Path) -> Dataset:
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

def format_record(example: dict) -> dict:
    role_map = {"system": "system", "human": "user", "gpt": "assistant"}
    messages = [
        {"role": role_map[turn["from"]], "content": turn["value"]}
        for turn in example["conversations"]
    ]
    return {
        "text": apply_chat_template_safe(tokenizer, messages, tokenize=False, add_generation_prompt=False)
    }

def infer_lora_target_modules(model) -> list[str]:
    common_suffixes = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "qkv_proj", "gate_up_proj", "gate_proj", "up_proj", "down_proj",
        "query_key_value", "c_attn", "c_proj", "dense", "fc1", "fc2",
    ]
    matches = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        suffix = name.split(".")[-1]
        if suffix in common_suffixes:
            matches.append(suffix)
    ordered = [suffix for suffix in common_suffixes if suffix in matches]
    if not ordered:
        raise RuntimeError("Unable to infer LoRA target modules for this architecture.")
    return ordered

def find_latest_checkpoint(path: Path):
    checkpoints = []
    for candidate in path.glob("checkpoint-*"):
        if candidate.is_dir():
            try:
                step = int(candidate.name.split("-", 1)[1])
            except (IndexError, ValueError):
                continue
            checkpoints.append((step, candidate))
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda item: item[0])
    return checkpoints[-1][1]

def choose_training_dtype() -> torch.dtype:
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16

training_plan = {
    "requested_model_key": BASE_MODEL_KEY,
    "selected_model_key": SELECTED_MODEL_KEY,
    "model_name": profile["model_name"],
    "access_url": profile["access_url"],
    "output_dir": str(OUTPUT_DIR),
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": profile["learning_rate"],
    "max_seq_length": profile["max_seq_length"],
    "gradient_accumulation_steps": profile["gradient_accumulation_steps"],
    "packing": PACKING,
    "push_adapter_to_hub": PUSH_ADAPTER_TO_HUB,
    "merge_after_train": MERGE_AFTER_TRAIN,
    "resume_from_checkpoint": str(find_latest_checkpoint(OUTPUT_DIR)) if find_latest_checkpoint(OUTPUT_DIR) else None,
}
print(json.dumps(training_plan, indent=2))

if START_TRAINING_NOW:
    train_dataset = load_jsonl_dataset(TRAIN_SPLIT_JSONL)
    eval_dataset = load_jsonl_dataset(EVAL_SPLIT_JSONL)
    if MAX_TRAIN_SAMPLES and len(train_dataset) > MAX_TRAIN_SAMPLES:
        train_dataset = train_dataset.select(range(MAX_TRAIN_SAMPLES))

    train_dataset = train_dataset.map(format_record, remove_columns=train_dataset.column_names)
    eval_dataset = eval_dataset.map(format_record, remove_columns=eval_dataset.column_names)

    compute_dtype = choose_training_dtype()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        profile["model_name"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=compute_dtype,
        token=HF_TOKEN or None,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()

    lora_targets = infer_lora_target_modules(model)
    print("LoRA target modules:", lora_targets)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        target_modules=lora_targets,
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=profile["gradient_accumulation_steps"],
        learning_rate=profile["learning_rate"],
        weight_decay=0.01,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        bf16=compute_dtype == torch.bfloat16,
        fp16=compute_dtype == torch.float16,
        optim="paged_adamw_8bit",
        seed=SEED,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        report_to=[] if REPORT_TO == "none" else [REPORT_TO],
        gradient_checkpointing=True,
        disable_tqdm=False,
        remove_unused_columns=False,
        dataset_text_field="text",
        max_length=profile["max_seq_length"],
        packing=PACKING,
        dataset_num_proc=1,
        logging_first_step=True,
        save_safetensors=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    latest_checkpoint = find_latest_checkpoint(OUTPUT_DIR)
    train_result = trainer.train(
        resume_from_checkpoint=str(latest_checkpoint) if latest_checkpoint else None
    )
    print(json.dumps(train_result.metrics, indent=2, default=str))

    adapter_path = OUTPUT_DIR / "adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"Adapter saved to {adapter_path}")

    if PUSH_ADAPTER_TO_HUB:
        if not HF_MODEL_REPO or not HF_TOKEN:
            raise RuntimeError("Set HF_MODEL_REPO and HF_TOKEN before pushing adapters.")
        api = HfApi(token=HF_TOKEN or None)
        api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN or None)
        api.upload_folder(
            folder_path=str(adapter_path),
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            path_in_repo="adapter",
            commit_message="Add meeting adapter",
            token=HF_TOKEN or None,
        )
        print(f"Adapter pushed to https://huggingface.co/{HF_MODEL_REPO}")

    if MERGE_AFTER_TRAIN:
        merged_path = OUTPUT_DIR / "merged"
        merged_model = model.merge_and_unload()
        merged_model.save_pretrained(merged_path, safe_serialization=True, max_shard_size="5GB")
        tokenizer.save_pretrained(merged_path)
        print(f"Merged model saved to {merged_path}")
else:
    print("Training not started. Set START_TRAINING_NOW = True and re-run this cell when ready.")

In [ ]:
RUN_SMOKE_TEST = False
SMOKE_TEST_SOURCE = "adapter"  # adapter | merged
SMOKE_TEST_PROMPT = "Alice: We need the Q3 budget finalized by Friday. Bob: I can have the numbers ready Thursday. Alice: Great. Also, please send the revised hiring plan. Bob: I will send it tonight.\n\nSummarize this meeting, list decisions, and extract action items."

if RUN_SMOKE_TEST:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if SMOKE_TEST_SOURCE == "merged":
        model_path = OUTPUT_DIR / "merged"
        if not model_path.exists():
            raise FileNotFoundError("Merged model not found. Run training with MERGE_AFTER_TRAIN = True first.")
        smoke_model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
        )
        smoke_tokenizer = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
        smoke_tokenizer = prepare_tokenizer_for_chat(smoke_tokenizer)
    else:
        adapter_path = OUTPUT_DIR / "adapter"
        if not adapter_path.exists():
            raise FileNotFoundError("Adapter not found. Run the training cell first.")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        smoke_tokenizer = AutoTokenizer.from_pretrained(profile["model_name"], token=HF_TOKEN, trust_remote_code=True)
        smoke_tokenizer = prepare_tokenizer_for_chat(smoke_tokenizer)
        smoke_model = AutoModelForCausalLM.from_pretrained(
            profile["model_name"],
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            token=HF_TOKEN,
        )
        smoke_model = PeftModel.from_pretrained(smoke_model, str(adapter_path))

    smoke_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": SMOKE_TEST_PROMPT},
    ]
    inputs = apply_chat_template_safe(
        smoke_tokenizer,
        smoke_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(smoke_model.device)

    with torch.no_grad():
        outputs = smoke_model.generate(
            inputs,
            max_new_tokens=300,
            do_sample=False,
            eos_token_id=smoke_tokenizer.eos_token_id,
            pad_token_id=smoke_tokenizer.pad_token_id,
        )

    generated = outputs[0][inputs.shape[1]:]
    print(smoke_tokenizer.decode(generated, skip_special_tokens=True))
else:
    print("Smoke test not run. Set RUN_SMOKE_TEST = True after adapter or merged weights exist.")

## Evaluation

This section gives you metrics that fit the way this model is trained.

- `ROUGE-1`, `ROUGE-2`, and `ROUGE-L` are appropriate for the summarization parts of this chat-SFT setup.
- `token_f1` is a simple overlap metric that is useful for comparing runs on held-out references.
- `action_item_f1` is only computed on prompts that explicitly ask for action items or decisions, where extraction-style overlap matters more than stylistic variation.

These metrics do not prove the model is perfect, but they are valid for comparing meeting variants trained with this notebook.

In [ ]:
RUN_FULL_EVAL = False
EVAL_SOURCE = "adapter"  # adapter | merged
EVAL_MAX_SAMPLES = 50
EVAL_ARTIFACT_PATH = ARTIFACTS_DIR / f"meeting_eval_{SELECTED_MODEL_KEY}.jsonl"

from collections import Counter, defaultdict
from peft import PeftModel
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def token_f1(reference: str, prediction: str) -> float:
    ref_tokens = normalize_text(reference).split()
    pred_tokens = normalize_text(prediction).split()
    if not ref_tokens or not pred_tokens:
        return 0.0
    overlap = sum((Counter(ref_tokens) & Counter(pred_tokens)).values())
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def split_items(text: str) -> list[str]:
    items = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        line = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line)
        line = re.sub(r"^(?:action items?|decisions?)\s*:\s*", "", line, flags=re.IGNORECASE)
        normalized = normalize_text(line)
        if normalized:
            items.append(normalized)
    return items

def item_f1(reference: str, prediction: str) -> float:
    ref_items = split_items(reference)
    pred_items = split_items(prediction)
    if not ref_items or not pred_items:
        return 0.0
    ref_set = set(ref_items)
    pred_set = set(pred_items)
    overlap = len(ref_set & pred_set)
    precision = overlap / len(pred_set)
    recall = overlap / len(ref_set)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def load_meeting_eval_rows(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_eval_model(eval_source: str):
    if eval_source == "merged":
        model_path = OUTPUT_DIR / "merged"
        if not model_path.exists():
            raise FileNotFoundError("Merged model not found. Run training with MERGE_AFTER_TRAIN = True first.")
        eval_model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
        )
        eval_tokenizer = AutoTokenizer.from_pretrained(str(model_path), trust_remote_code=True)
        eval_tokenizer = prepare_tokenizer_for_chat(eval_tokenizer)
    else:
        adapter_path = OUTPUT_DIR / "adapter"
        if not adapter_path.exists():
            raise FileNotFoundError("Adapter not found. Run the training cell first.")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        eval_tokenizer = AutoTokenizer.from_pretrained(profile["model_name"], token=HF_TOKEN or None, trust_remote_code=True)
        eval_tokenizer = prepare_tokenizer_for_chat(eval_tokenizer)
        base_model = AutoModelForCausalLM.from_pretrained(
            profile["model_name"],
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            token=HF_TOKEN or None,
        )
        eval_model = PeftModel.from_pretrained(base_model, str(adapter_path))
    return eval_model, eval_tokenizer

def generate_meeting_response(eval_model, eval_tokenizer, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    inputs = apply_chat_template_safe(
        eval_tokenizer,
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(eval_model.device)
    with torch.no_grad():
        outputs = eval_model.generate(
            inputs,
            max_new_tokens=400,
            do_sample=False,
            eos_token_id=eval_tokenizer.eos_token_id,
            pad_token_id=eval_tokenizer.pad_token_id,
        )
    generated = outputs[0][inputs.shape[1]:]
    return eval_tokenizer.decode(generated, skip_special_tokens=True).strip()

if RUN_FULL_EVAL:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    eval_rows = load_meeting_eval_rows(EVAL_SPLIT_JSONL)[:EVAL_MAX_SAMPLES]
    eval_model, eval_tokenizer = load_eval_model(EVAL_SOURCE)

    rouge_totals = defaultdict(float)
    token_f1_scores = []
    action_item_f1_scores = []
    results = []

    for row in tqdm(eval_rows, desc="Meeting evaluation"):
        prompt = row["conversations"][1]["value"]
        reference = row["conversations"][2]["value"]
        prediction = generate_meeting_response(eval_model, eval_tokenizer, prompt)

        rouge_scores = scorer.score(reference, prediction)
        rouge_totals["rouge1"] += rouge_scores["rouge1"].fmeasure
        rouge_totals["rouge2"] += rouge_scores["rouge2"].fmeasure
        rouge_totals["rougeL"] += rouge_scores["rougeL"].fmeasure

        tf1 = token_f1(reference, prediction)
        token_f1_scores.append(tf1)

        prompt_lower = prompt.lower()
        action_f1 = None
        if any(marker in prompt_lower for marker in ["action item", "action items", "key decisions", "list decisions"]):
            action_f1 = item_f1(reference, prediction)
            action_item_f1_scores.append(action_f1)

        results.append({
            "prompt": prompt,
            "reference": reference,
            "prediction": prediction,
            "rouge1_f1": rouge_scores["rouge1"].fmeasure,
            "rouge2_f1": rouge_scores["rouge2"].fmeasure,
            "rougeL_f1": rouge_scores["rougeL"].fmeasure,
            "token_f1": tf1,
            "action_item_f1": action_f1,
        })

    with EVAL_ARTIFACT_PATH.open("w", encoding="utf-8") as handle:
        for result in results:
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")

    count = len(results)
    summary = {
        "model_name": profile["model_name"],
        "selected_model_key": SELECTED_MODEL_KEY,
        "eval_source": EVAL_SOURCE,
        "eval_samples": count,
        "rouge1_f1_mean": rouge_totals["rouge1"] / count if count else 0.0,
        "rouge2_f1_mean": rouge_totals["rouge2"] / count if count else 0.0,
        "rougeL_f1_mean": rouge_totals["rougeL"] / count if count else 0.0,
        "token_f1_mean": sum(token_f1_scores) / len(token_f1_scores) if token_f1_scores else 0.0,
        "action_item_f1_mean": sum(action_item_f1_scores) / len(action_item_f1_scores) if action_item_f1_scores else None,
        "artifact_path": str(EVAL_ARTIFACT_PATH),
    }
    print(json.dumps(summary, indent=2))
else:
    print("Evaluation not run. Set RUN_FULL_EVAL = True after adapter or merged weights exist.")